In [ ]:
!mkdir -p models/
!mkdir -p data/ezpz2/
!wget https://files.krazyorange.dev/data/ezpz2/ezpz2.corptok.txt -O data/ezpz2/ezpz2.corptok.txt
!wget https://files.krazyorange.dev/data/ezpz2/ezpz2.vocab.json -O data/ezpz2/ezpz2.vocab.json


In [ ]:
# file: corpus.py

from dataclasses import dataclass


@dataclass
class LLNCACorpusConfig:
    trunc_ratio: float = 0.8
    trunc_split: str = " "


class LLNCACorpus:
    def __init__(self, config: LLNCACorpusConfig):
        self.config = config

    def generate(
        self,
        in_file: str,
        out_file: str | None = None,
    ):
        if out_file is None:
            parts = in_file.split(".")
            out_file = ".".join(parts[:-1]) + ".corp" + "." + parts[-1]

        print(f"\033[2m  in: {in_file}\033[0m")
        with (
            open(in_file, "r") as _in_file,
            open(out_file, "w") as _out_file,
        ):
            trunc_split = self.config.trunc_split
            for line in _in_file:
                line = line.strip()
                words = line.split(trunc_split)
                trunc_i = int(len(words) * self.config.trunc_ratio)
                words_x = trunc_split.join(words[:trunc_i])
                words_y = trunc_split.join(words[trunc_i:])
                _out_file.write(f"{words_x}{trunc_split}\n{words_y}\n")

        print(f"\033[2m out: {out_file}\033[0m")


if __name__ == "__main__DISABLED":
    print("loading config...")
    config = LLNCACorpusConfig()
    corpus = LLNCACorpus(config=config)
    print("generating corpus...")
    corpus.generate(in_file="data/ezpz2/ezpz2.txt")

In [ ]:
# file: tokenizer.py

import json
import os
import time
from collections import Counter
from concurrent.futures import ProcessPoolExecutor
from itertools import pairwise

from tqdm import tqdm


def _count_pairwise(text: str) -> Counter:
    return Counter(pairwise(text))


def print_ttlr(text: str, lim: int = 120, dim=True):
    r_text = repr(text)[1:-1]
    l_r_text = len(r_text)
    l_r_r_text = len(r_text.replace("\\n", ""))
    tqdm.write(
        ("\033[2m" if dim else "")
        + f"[{l_r_r_text}] {r_text[:lim]}"
        + ("..." if l_r_text > lim else "")
        + ("\033[22m" if dim else "")
    )


type LLNCAVocab = dict[int, str | tuple[str, str]]
type LLNCAIVocab = dict[str | tuple[str, str], int]


class LLNCATokenizer:
    def __init__(self, vocab: LLNCAVocab | None = None):
        self.vocab: LLNCAVocab = vocab if vocab is not None else {}

        self.ivocab: LLNCAIVocab = {}
        self.build_ivocab()
        self.ivocab_dirty = False

        self.next_tok = max(255, max(self.vocab.keys())) + 1 if self.vocab else 256

        self.n_workers = os.cpu_count() or 8
        self.executor = ProcessPoolExecutor(max_workers=self.n_workers)

        self.vocab[0] = "\x00"
        for i in range(32, 127):
            self.vocab[i] = chr(i)

    def load_vocab(self, vocab):
        self.vocab = vocab
        self.vocab = {
            (int(k) if isinstance(k, str) else k): (
                (v[0], v[1]) if isinstance(v, list) else v
            )
            for k, v in self.vocab.items()
        }

        self.build_ivocab()
        self.ivocab_dirty = False

        max_tok = max(self.vocab.keys())
        self.next_tok = max_tok + 1

    def train(self, text: str, debug: bool = False):
        prev = time.perf_counter()
        n_merges = self._train(text, debug)
        curr = time.perf_counter()
        diff = curr - prev
        print(f"\033[2mmade {n_merges} merges in {diff:.6f} seconds.\033[0m")

    def _train(self, text: str, debug: bool = False):
        self.ivocab_dirty = True

        chars_unique = set(text)
        for char in chars_unique:
            tok = next(iter(char.encode("utf-8")))
            self.vocab[tok] = char
        self.vocab[0] = "\x00"

        toks_str = text.encode("utf-8").decode("latin-1")
        if debug:
            print_ttlr(toks_str)

        n_merges = 0
        # with tqdm(dynamic_ncols=True, leave=False, unit="merges") as pbar:
        while len(toks_str) > 1:
            counts = self.count_pairs(toks_str)
            # tqdm.write(str(counts))
            if not counts:
                break

            # most_common = counts.most_common(1)[0]
            most_common = (("", ""), 0)
            for pair, count in counts.most_common():
                if count < most_common[1]:
                    break
                if "\n" not in pair:  # and " " not in pair:
                    most_common = (pair, count)

            most_pair, most_count = most_common
            if most_count <= 1:
                break

            self.vocab[self.next_tok] = most_pair
            c1, c2 = most_pair
            toks_str = toks_str.replace(c1 + c2, chr(self.next_tok))

            self.next_tok += 1
            n_merges += 1
            if debug:
                print_ttlr(toks_str)
            # pbar.update()

        return n_merges

    def encode(self, text: str, debug: bool = False):
        prev = time.perf_counter()
        toks_str, n_merges = self._encode(text, debug)
        curr = time.perf_counter()
        diff = curr - prev
        print(f"\033[2mmade {n_merges} merges in {diff:.6f} seconds.\033[0m")
        return toks_str

    def _encode(self, text: str, debug: bool = False):
        if self.ivocab_dirty:
            self.build_ivocab()
            self.ivocab_dirty = False

        toks_str = text.encode("utf-8").decode("latin-1")
        if debug:
            print_ttlr(toks_str)

        n_merges = 0
        while len(toks_str) > 1:
            counts = self.count_pairs(toks_str)
            if not counts:
                break

            most_tok = 0
            most_common = (("", ""), 0)
            for pair, count in counts.most_common():
                if pair in self.ivocab:
                    most_tok = self.ivocab[pair]
                    most_common = (pair, count)
                    break

            most_pair, most_count = most_common
            if most_count <= 1:
                break

            c1, c2 = most_pair
            toks_str = toks_str.replace(c1 + c2, chr(most_tok))
            n_merges += 1

            if debug:
                print_ttlr(toks_str)

        return toks_str, n_merges

    def count_pairs(self, toks_str: str) -> Counter[tuple[str, str]]:
        n_toks = len(toks_str)

        if n_toks < 100_000:
            return Counter(pairwise(toks_str))

        buf_sz = n_toks // self.n_workers
        bufs = []

        for i in range(self.n_workers):
            start = i * buf_sz
            end = start + buf_sz + (1 if i < self.n_workers - 1 else 0)
            bufs.append(toks_str[start:end])

        counts = Counter()
        for _counts in self.executor.map(_count_pairwise, bufs):
            counts.update(_counts)

        return counts

    def decode(self, tok_str: str):
        return "".join([self.expand_tok(ord(tok)) for tok in tok_str])

    def expand_tok(self, tok: int):
        toks: list[int] = [tok]
        strs: list[str] = []
        expanded = True
        while expanded:
            new_toks: list[int] = []
            expanded = False
            for _tok in toks:
                sub = self.vocab[_tok]
                if isinstance(sub, tuple):
                    new_toks.extend([ord(sub_e) for sub_e in sub])
                    expanded = True
                elif isinstance(sub, str):
                    new_toks.append(_tok)
            toks = new_toks
        for _tok in toks:
            sub = self.vocab[_tok]
            if isinstance(sub, str):
                strs.append(sub)
            elif isinstance(sub, tuple):
                raise TypeError("token didn't expand to str as expected")
        return "".join(strs)

    def build_ivocab(self):
        self.ivocab = {v: k for k, v in self.vocab.items()}

    def __len__(self):
        return len(self.vocab)

    def __del__(self):
        self.executor.shutdown(wait=False)


def main():
    corp_path = "data/ezpz2/ezpz2.corp.txt"
    corptok_path = "data/ezpz2/ezpz2.corptok.txt"
    vocab_path = "data/ezpz2/ezpz2.vocab.json"

    print("loading text...", end=" ")

    corp = """"""
    with open(corp_path, "r") as file:
        corp = file.read()
    print(f"\033[2m{corp_path}\033[0m")

    tokenizer = LLNCATokenizer()

    print("training...", end=" ")
    tokenizer.train(corp)

    # print("loading vocab...", end=" ")
    # with open(vocab_path, "r") as file:
    #     tokenizer.load_vocab(json.load(file))
    # print(f"\033[2m{vocab_path}\033[0m")

    # print(tokenizer.vocab)

    print("encoding...", end=" ")
    corptok = tokenizer.encode(corp)

    print("writing...", end=" ")
    with open(corptok_path, "w") as file:
        file.write(corptok)
    print(f"\033[2m{corptok_path}\033[0m")

    print("saving...", end=" ")
    with open(vocab_path, "w") as file:
        json.dump(tokenizer.vocab, file)
    print(f"\033[2m{vocab_path}\033[0m")

    return tokenizer


if __name__ == "__main__DISABLED":
    main()

In [ ]:
# file: dataset.py

import math
import random
from collections import defaultdict
from dataclasses import dataclass

from torch.utils.data import Dataset, Sampler


@dataclass
class LLNCADatasetConfig:
    file: str


class LLNCADataset(Dataset):
    def __init__(self, config: LLNCADatasetConfig):
        self.config = config
        self.rows = []
        self.load()

    def load(self):
        with open(self.config.file) as file:
            lines = file.readlines()
        lines = [line.rstrip("\n") for line in lines]
        for i in range(0, len(lines), 2):
            x = lines[i][:]
            y = lines[i][:] + lines[i + 1][:]
            self.rows.append((x, y))

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx: int):
        return self.rows[idx]


@dataclass
class LLNCADataSamplerConfig:
    bin_interval: int
    batch_len: int
    drop_last: bool
    shuffle: bool


class LLNCADataSampler(Sampler):
    def __init__(
        self, dataset: LLNCADataset, config: LLNCADataSamplerConfig, debug: bool = False
    ):
        self.dataset = dataset
        self.config = config

        self.bins = defaultdict(list)
        b = self.config.bin_interval
        for i in range(len(dataset)):
            l = len(dataset[i][1])
            l = math.ceil(l / b) * b
            self.bins[l].append(i)

        if debug:
            print("\033[2mbins:\033[0m")
            bin_sz_maxlen = max(len(str(bin_sz)) for bin_sz in self.bins)
            for bin_sz, bin_l in sorted(self.bins.items()):
                bin_sz_str = str(bin_sz).rjust(bin_sz_maxlen)
                print(f"  \033[2m{bin_sz_str}: {'.' * len(bin_l)}\033[0m")

    def __iter__(self):
        batches = []

        for idxs in self.bins.values():
            if self.config.shuffle:
                random.shuffle(idxs)

            for i in range(0, len(idxs), self.config.batch_len):
                batch = idxs[i : i + self.config.batch_len]
                if self.config.drop_last and len(batch) < self.config.batch_len:
                    continue
                batches.append(batch)

        if self.config.shuffle:
            random.shuffle(batches)

        for batch in batches:
            yield batch

    def __len__(self):
        tn_batches = 0
        for idxs in self.bins.values():
            n_batches = len(idxs) // self.config.batch_len
            if not self.config.drop_last and len(idxs) % self.config.batch_len != 0:
                n_batches += 1
            tn_batches += n_batches
        return tn_batches


if __name__ == "__main__DISABLED":
    corptok_path = "data/ezpz2/ezpz2.corptok.txt"
    print(f"loading dataset... \033[2m{corptok_path}\033[0m")
    dataset_config = LLNCADatasetConfig(file=corptok_path)
    dataset = LLNCADataset(dataset_config)
    print("loading sampler...")
    sampler_config = LLNCADataSamplerConfig(
        bin_interval=16,
        batch_len=16,
        drop_last=False,
        shuffle=True,
    )
    sampler = LLNCADataSampler(dataset, sampler_config, debug=True)

In [ ]:
# file: nca.py

from dataclasses import dataclass

import torch
from torch import nn
from torch import optim
from torch.nn import functional as F


class LLNCAFilter(nn.Module):
    def __init__(self, in_channels: int):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = in_channels * 3
        self.conv = nn.Conv1d(
            self.in_channels,
            self.out_channels,
            kernel_size=2,
            groups=in_channels,
        )
        self.reset()

    def reset(self):
        identity = torch.tensor([0.0, 1.0])
        previous = torch.tensor([1.0, 0.0])
        gradient = torch.tensor([-1.0, 1.0])
        kernel = torch.stack([identity, previous, gradient])[:, None, :]
        with torch.no_grad():
            self.conv.weight.copy_(kernel.repeat(self.in_channels, 1, 1))

    def forward(self, x):
        x = F.pad(x, (1, 0), mode="constant", value=0.0)
        return self.conv(x)


@dataclass
class LLNCANCAConfig:
    channels: int
    mlp_width: int
    mlp_depth: int
    activation_fn: str
    update_rate: float
    alive_threshold: float


class LLNCANCA(nn.Module):
    def __init__(self, config: LLNCANCAConfig):
        super().__init__()
        self.config = config

        self.filter = LLNCAFilter(self.config.channels)

        mlp_width = self.config.mlp_width
        filter_channels = self.filter.out_channels
        activation_fn: type[nn.Module] = getattr(nn, self.config.activation_fn)
        layers = []
        layers.append(nn.Conv1d(filter_channels, mlp_width, kernel_size=1))
        for _ in range(self.config.mlp_depth - 2):
            layers.append(nn.Conv1d(mlp_width, mlp_width, kernel_size=1))
            layers.append(activation_fn())
        layers.append(nn.Conv1d(mlp_width, self.config.channels, kernel_size=1))

        self.seq = nn.Sequential(*layers)

    def add_channels(self, x: torch.Tensor):
        b, c, w = x.shape
        n_h = self.config.channels - c
        h = torch.zeros((b, n_h, w), device=x.device)
        y = torch.cat([x, h], dim=1)
        return y

    def get_alive_mask(self, x: torch.Tensor):
        y = F.pad(x.abs(), (1, 0), mode="constant", value=0.0)
        y = F.max_pool1d(y, kernel_size=2, stride=1)
        y = y.amax(dim=1, keepdim=True)
        y = y >= self.config.alive_threshold
        return y.float()

    def get_update_mask(self, x: torch.Tensor):
        b, _, w = x.shape
        y = torch.rand(b, 1, w, device=x.device) < self.config.update_rate
        return y.float()

    def step(self, x: torch.Tensor, freeze_mask: torch.Tensor):
        y = self.filter(x)
        y = self.seq(y)

        alive_mask = self.get_alive_mask(x)
        update_mask = self.get_update_mask(x)
        y = y * freeze_mask * alive_mask * update_mask

        y = x + y
        return y

    def forward(
        self, x: torch.Tensor, steps: int = 1, freeze_mask: torch.Tensor | None = None
    ):
        if freeze_mask is None:
            freeze_mask = torch.ones_like(x)

        for i in range(steps):
            x = self.step(x, freeze_mask)

        return x


# nca img
# x.shape =
#   (B    , C      , W    )
#   (batch, channel, width)


# llnca adv nca gan
# use same nca setup
# frozen channels holding gen nca output
# output channel grades realism
# minimize average or wtvr

In [ ]:
# file: embedding.py

from dataclasses import dataclass

import numpy as np
import torch
from torch import nn



@dataclass
class LLNCAEmbeddingsConfig:
    n_dims: int


class LLNCAEmbeddings:
    def __init__(
        self,
        tokenizer: LLNCATokenizer,
        config: LLNCAEmbeddingsConfig,
        debug: bool = False,
    ):
        self.tokenizer = tokenizer
        self.config = config

        self.embeddings = nn.Embedding(
            num_embeddings=len(self.tokenizer),
            embedding_dim=self.config.n_dims,
            padding_idx=0,
        )

        self.tok_embed_map: dict[int, int] = {
            k: i for i, (k, v) in enumerate(sorted(self.tokenizer.vocab.items()))
        }
        self.i_tok_embed_map: dict[int, int] = {
            v: k for k, v in self.tok_embed_map.items()
        }
        self.tok_str_embed_map: dict[str, int] = {
            (k if isinstance(k, str) else k[0] + k[1]): self.tok_embed_map[v]
            for k, v in self.tokenizer.ivocab.items()
        }

        if debug:
            print("\033[2membeddings\033[0m")
            sorted_keys = sorted(self.tokenizer.vocab.keys())
            for tok in sorted_keys[:5]:
                self.print_embed(tok)
            print("  \033[2m...\033[0m")
            for tok in sorted_keys[-5:]:
                self.print_embed(tok)

            for char in "grass":
                self.print_embed_str(char)

    def embed_from_tok(self, tok: int):
        embed_i = self.tok_embed_map[tok]
        return self.embeddings(
            torch.tensor(embed_i, device=self.embeddings.weight.device)
        )

    def print_embed(self, tok: int):
        with np.printoptions(linewidth=10000, precision=4, suppress=True):
            embedding_i = self.tok_embed_map[tok]
            embedding = self.embeddings.weight[embedding_i].detach().numpy()
            print(f"  \033[2m[{tok}]\t{embedding}\033[0m")

    def print_embed_str(self, tok_str: str):
        with np.printoptions(linewidth=10000, precision=4, suppress=True):
            embedding_i = self.tok_str_embed_map[tok_str]
            embedding = self.embeddings.weight[embedding_i].detach().numpy()
            print(f"  \033[2m[{tok_str}]\t{embedding}\033[0m")

    def load(self, state):
        self.embeddings.load_state_dict(state)

    def save(self):
        return self.embeddings.state_dict()


if __name__ == "__main__DISABLED":
    tokenizer = main()
    print("loading embeddings...")
    config = LLNCAEmbeddingsConfig(n_dims=8)
    embedding = LLNCAEmbeddings(tokenizer=tokenizer, config=config, debug=True)

In [ ]:
# file: main.py

import json
import math
import random
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn.functional as F
from torch import optim
from tqdm.auto import tqdm
from visdom import Visdom



@dataclass
class LLNCAOptimConfig:
    lr: float
    lr_gamma: float
    weight_decay: float
    betas: tuple[float, float]


@dataclass
class LLNCAGenConfig:
    nca: LLNCANCAConfig
    optim: LLNCAOptimConfig
    steps: tuple[int, int]


@dataclass
class LLNCAAdvConfig:
    nca: LLNCANCAConfig
    optim: LLNCAOptimConfig
    steps: tuple[int, int]


@dataclass
class LLNCACheckpointingConfig:
    major_name: str
    minor_name: str
    folder: str
    freq: int


@dataclass
class LLNCAConfig:
    corpus: LLNCACorpusConfig
    dataset: LLNCADatasetConfig
    sampler: LLNCADataSamplerConfig
    embeddings: LLNCAEmbeddingsConfig
    vocab: LLNCAVocab
    gen: LLNCAGenConfig
    adv: LLNCAAdvConfig
    checkpointing: LLNCACheckpointingConfig
    n_epochs: int
    lambda_pxl: float
    lambda_gan: float


class LLNCA:
    def __init__(
        self,
        checkpoint: dict | None = None,
        config: LLNCAConfig | None = None,
        visdom: Visdom | None = None,
        debug: bool = False,
    ):
        if checkpoint is None and config is None:
            raise RuntimeError(
                "\033[31m[error]\033[0m either checkpoint or config must be provided."
            )

        if checkpoint is not None and config is not None:
            checkpoint["config"] = config

        self.config: LLNCAConfig = (
            config if checkpoint is None else checkpoint["config"]
        )  # type: ignore

        self.visdom = visdom

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        print("loading...", end=" ")

        self.corpus = LLNCACorpus(config=self.config.corpus)
        print("\033[2mcorpus\033[0m ", end=" ")

        self.tokenizer = LLNCATokenizer(vocab=self.config.vocab)
        print("\033[2mtokenizer\033[0m ", end=" ")

        self.embeddings = LLNCAEmbeddings(self.tokenizer, config=self.config.embeddings)
        self.embeddings.embeddings.to(self.device)
        print("\033[2membeddings\033[0m ", end=" ")

        self.dataset = LLNCADataset(config=self.config.dataset)
        print("\033[2mdataset\033[0m ", end=" ")

        self.sampler = LLNCADataSampler(self.dataset, config=self.config.sampler)
        print("\033[2msampler\033[0m ", end=" ")

        self.dataloader = torch.utils.data.DataLoader(
            dataset=self.dataset, batch_sampler=self.sampler
        )
        print("\033[2mdataloader\033[0m ", end=" ")

        self.gen_nca = LLNCANCA(config=self.config.gen.nca).to(self.device)
        # self.gen_nca = torch.compile(self.gen_nca)
        print("\033[2mgen_nca\033[0m ", end=" ")

        self.adv_nca = LLNCANCA(config=self.config.adv.nca).to(self.device)
        # self.adv_nca = torch.compile(self.adv_nca)
        print("\033[2madv_nca\033[0m ", end=" ")

        gen_optim_conf = self.config.gen.optim
        adv_optim_conf = self.config.adv.optim

        self.gen_optim = optim.AdamW(
            list(self.gen_nca.parameters())
            + list(self.embeddings.embeddings.parameters()),
            lr=gen_optim_conf.lr,
            betas=gen_optim_conf.betas,
            weight_decay=gen_optim_conf.weight_decay,
        )
        print("\033[2mgen_optim\033[0m ", end=" ")

        self.adv_optim = optim.AdamW(
            self.adv_nca.parameters(),
            lr=adv_optim_conf.lr,
            betas=adv_optim_conf.betas,
            weight_decay=adv_optim_conf.weight_decay,
        )
        print("\033[2madv_optim\033[0m ", end=" ")

        self.gen_scheduler = optim.lr_scheduler.ExponentialLR(
            self.gen_optim, gen_optim_conf.lr_gamma
        )
        print("\033[2mgen_scheduler\033[0m ", end=" ")

        self.adv_scheduler = optim.lr_scheduler.ExponentialLR(
            self.adv_optim, adv_optim_conf.lr_gamma
        )
        print("\033[2madv_scheduler\033[0m ", end=" ")

        print()

        if checkpoint is not None:
            self.embeddings.embeddings.load_state_dict(checkpoint["embeddings"])
            self.gen_nca.load_state_dict(checkpoint["gen_nca"])
            self.adv_nca.load_state_dict(checkpoint["adv_nca"])
            self.gen_optim.load_state_dict(checkpoint["gen_optim"])
            self.adv_optim.load_state_dict(checkpoint["adv_optim"])
            self.gen_scheduler.load_state_dict(checkpoint["gen_scheduler"])
            self.adv_scheduler.load_state_dict(checkpoint["adv_scheduler"])

        self.curr_epoch = 0

    def get_state(self):
        state = {
            "config": self.config,
            "embeddings": self.embeddings.embeddings.state_dict(),
            "gen_nca": self.gen_nca.state_dict(),
            "adv_nca": self.adv_nca.state_dict(),
            "gen_optim": self.gen_optim.state_dict(),
            "adv_optim": self.adv_optim.state_dict(),
            "gen_scheduler": self.gen_scheduler.state_dict(),
            "adv_scheduler": self.adv_scheduler.state_dict(),
            "curr_epoch": self.curr_epoch,
        }
        return state

    def save(self, path):
        state = self.get_state()
        torch.save(state, path)

    def checkpoint(self):
        segments = [
            self.config.checkpointing.folder,
            "/" if not self.config.checkpointing.folder.endswith("/") else "",
            self.config.checkpointing.major_name,
            "-" if self.config.checkpointing.minor_name != "" else "",
            self.config.checkpointing.minor_name,
            "-",
            str(self.curr_epoch + 1),
            ".pth",
        ]
        path = "".join(segments)
        self.save(path)

    def train(self):
        self.gen_nca.train()
        self.adv_nca.train()

        for epoch_i in tqdm(
            range(self.curr_epoch, self.config.n_epochs),
            total=self.config.n_epochs,
            leave=False,
            dynamic_ncols=True,
            unit="epoch",
        ):
            self.curr_epoch = epoch_i
            loss_acc = 0

            for x, y in self.dataloader:
                n_steps = random.randint(
                    self.config.gen.steps[0], self.config.gen.steps[1]
                )
                xs, ys = self.tok_to_embed(x, y)
                xs = self.gen_nca.add_channels(xs)
                ys_pred = self.gen_nca(xs, steps=n_steps)
                ys_pred = ys_pred[:, : self.config.embeddings.n_dims, :]

                loss = F.l1_loss(ys_pred, ys.detach())
                loss_acc += loss.item()
                loss.backward()
                self.gen_optim.step()
                self.gen_optim.zero_grad()

            self.gen_scheduler.step()

            loss_avg = loss_acc / len(self.dataloader)
            if self.visdom:
                if epoch_i == 0:
                    self.loss_win = self.visdom.line(
                        X=np.array([0]),
                        Y=np.array([loss_avg]),
                        opts={"title": "loss", "xlabel": "epochs", "ylabel": "loss"},
                    )
                else:
                    self.visdom.line(
                        X=np.array([epoch_i]),
                        Y=np.array([loss_avg]),
                        win=self.loss_win,
                        update="append",
                    )

            if (epoch_i + 1) % self.config.checkpointing.freq == 0:
                self.checkpoint()

    def eval(self):
        self.gen_nca.eval()
        self.adv_nca.eval()
        for x, y in self.dataloader:
            n_steps = (self.config.gen.steps[0] + self.config.gen.steps[1]) // 2
            xs, ys = self.tok_to_embed(x, y)
            xs = self.gen_nca.add_channels(xs)
            y_pred = y
            for _ in range(n_steps):
                xs = self.gen_nca(xs, steps=1)
                ys_pred = xs[:, : self.config.embeddings.n_dims, :]
                idxs, _ = self.nearest_embed(ys_pred)
                y_pred = self.reconstruct_str(idxs)
                yield y_pred[0]

    def reconstruct_str(self, idxs: torch.Tensor):
        idxs = idxs.cpu()
        strs = []
        for i in range(len(idxs)):
            chrs = []
            for n in idxs[i]:
                _ord = self.embeddings.i_tok_embed_map[int(n.item())]
                if _ord == 0:
                    continue
                _chr = chr(_ord)
                chrs.append(_chr)
            _str = "".join(chrs)
            strs.append(_str)
        return strs

    def tok_to_embed(self, x_strs: list[str], y_strs: list[str]):
        bin_size = self.config.sampler.bin_interval
        max_len = max(max(len(s) for s in x_strs), max(len(s) for s in y_strs))
        pad_len = math.ceil(max_len / bin_size) * bin_size
        tok_map = self.embeddings.tok_embed_map

        x_idxs = [
            [tok_map.get(ord(c), 0) for c in s] + [0] * (pad_len - len(s))
            for s in x_strs
        ]
        y_idxs = [
            [tok_map.get(ord(c), 0) for c in s] + [0] * (pad_len - len(s))
            for s in y_strs
        ]

        x_idx_tensor = torch.tensor(x_idxs, dtype=torch.long, device=self.device)
        y_idx_tensor = torch.tensor(y_idxs, dtype=torch.long, device=self.device)

        xs = self.embeddings.embeddings(x_idx_tensor).permute(0, 2, 1)
        ys = self.embeddings.embeddings(y_idx_tensor).permute(0, 2, 1)

        return xs, ys

    def nearest_embed(self, x: torch.Tensor):
        w = self.embeddings.embeddings.weight
        x = x.permute(0, 2, 1)
        x_norm = F.normalize(x, p=2, dim=-1)
        w_norm = F.normalize(w, p=2, dim=-1)
        cosine = torch.matmul(x_norm, w_norm.T)
        idxs = torch.argmax(cosine, dim=-1)
        y = self.embeddings.embeddings(idxs).permute(0, 2, 1)
        return idxs, y


if __name__ == "__main__":
    corptok_path = "data/ezpz2/ezpz2.corptok.txt"
    vocab_path = "data/ezpz2/ezpz2.vocab.json"

    corpus_config = LLNCACorpusConfig(trunc_ratio=0.7, trunc_split=" ")
    dataset_config = LLNCADatasetConfig(file=corptok_path)
    sampler_config = LLNCADataSamplerConfig(
        bin_interval=16,
        batch_len=16,
        drop_last=False,
        shuffle=True,
    )
    embedding_config = LLNCAEmbeddingsConfig(
        n_dims=8,
    )

    tokenizer = LLNCATokenizer()
    with open(vocab_path) as file:
        tokenizer.load_vocab(json.load(file))
    vocab = tokenizer.vocab

    gen_config = LLNCAGenConfig(
        LLNCANCAConfig(
            channels=64,
            mlp_width=256,
            mlp_depth=16,
            activation_fn="ReLU",
            update_rate=0.5,
            alive_threshold=0.01,
        ),
        LLNCAOptimConfig(
            lr=1e-3,
            lr_gamma=0.9998,
            weight_decay=0.01,
            betas=(0.9, 0.95),
        ),
        steps=(20, 30),
    )

    adv_config = LLNCAAdvConfig(
        LLNCANCAConfig(
            channels=64,
            mlp_width=256,
            mlp_depth=16,
            activation_fn="ReLU",
            update_rate=0.5,
            alive_threshold=0.01,
        ),
        LLNCAOptimConfig(
            lr=1e-3,
            lr_gamma=0.9998,
            weight_decay=0.01,
            betas=(0.9, 0.95),
        ),
        steps=(20, 30),
    )

    checkpointing_config = LLNCACheckpointingConfig(
        major_name="alpha",
        minor_name="b",
        folder="models",
        freq=1000,
    )

    config = LLNCAConfig(
        corpus=corpus_config,
        dataset=dataset_config,
        sampler=sampler_config,
        embeddings=embedding_config,
        vocab=vocab,
        gen=gen_config,
        adv=adv_config,
        checkpointing=checkpointing_config,
        n_epochs=10000,
        lambda_pxl=0.5,
        lambda_gan=0.5,
    )

    visdom = Visdom(server="https://visdom.krazyorange.dev", port=443)
    if not visdom.check_connection():
        visdom = None

    llnca = LLNCA(config=config, visdom=visdom)
    llnca.train()

In [ ]:
# file: eval.py

import sys
import time

import torch


if __name__ == "__main__DISABLED":
    if len(sys.argv) < 2:
        print("\033[91merror:\033[0m model path required")
        sys.exit(1)

    path = sys.argv[1]

    llnca = LLNCA(
        checkpoint=torch.load(
            path,
            weights_only=False,
            map_location=torch.device("cpu"),
        )
    )

    llnca1 = LLNCA(
        checkpoint=torch.load(
            "models/alpha-a-25.pth",
            weights_only=False,
            map_location=torch.device("cpu"),
        )
    )
    llnca2 = LLNCA(
        checkpoint=torch.load(
            "models/alpha-a-1000.pth",
            weights_only=False,
            map_location=torch.device("cpu"),
        )
    )
    embds1 = llnca1.embeddings.embeddings.weight
    embds2 = llnca2.embeddings.embeddings.weight
    diff = embds2 - embds1
    print(diff)
    print("min", diff.min())
    print("max", diff.max())
    print("mean", diff.mean())
    print("std", diff.std())

    while True:
        prev_frame = ""
        for frame in llnca.eval():
            frame_str = llnca.tokenizer.decode(frame)
            print("\33[2K\r" + frame_str, end="", flush=True)
            if prev_frame != frame_str:
                time.sleep(0.25)
            prev_frame = frame_str